In [45]:
import pandas as pd

# temporary formatting purposes
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_colwidth', None)

In [46]:
data = pd.read_csv("professors_info.csv")

<h1>Data Pre-Processing</h1>

Rename columns for clarity and understanding

`SID` --> `School_ID`
<br>
`Class Taken` --> `Course Code`

In [49]:
data.rename(columns={"SID": "School_ID", "Class Taken": "Course Code"}, inplace=True)
data.head()

,School_ID,School Name,Professor Name,Overall Quality,Num Reviews,Department,Course Code,Review Date,Quality,Difficulty,Review Text
0,1147,Washington University in St. Louis,Karl Schaefer,4.8,20,Mathematics,MATH233,"Sep 6th, 2024",5.0,2.0,Best Professor I've Ever Had. He explains comp...
1,1147,Washington University in St. Louis,Karl Schaefer,4.8,20,Mathematics,MATH233,"Jun 14th, 2024",5.0,2.0,Probably one of the best math professors at wash
2,1147,Washington University in St. Louis,Karl Schaefer,4.8,20,Mathematics,MATH310W,"Jun 4th, 2024",5.0,2.0,Schaefer is a very skilled educator and he is ...
3,1147,Washington University in St. Louis,Karl Schaefer,4.8,20,Mathematics,MATH233,"Jun 4th, 2024",5.0,2.0,Schaefer remains unparalleled for Calculus lec...
4,1147,Washington University in St. Louis,Karl Schaefer,4.8,20,Mathematics,MATH233,"May 22nd, 2024",5.0,2.0,Professor Schaefer is an amazing Professor. He...


Convert all `review dates` from strings to datetime

Rate My Professor lists review dates in ordinal format (i.e. November 1st, November 2nd, November 3rd, November 4th, 2024). So we need to get rid of the ordinal suffix and convert to datetime.

In [51]:
# removes ordinal suffixes (-st, -nd, -rd, -th)
data['Review Date'] = data['Review Date'].str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True)

# convert to datetime
data['Review Date'] = pd.to_datetime(data['Review Date'], format='%b %d, %Y')

Drop the `Num Reviews` column from the dataset. This is a redundant column since we can very easily query this result.

In [53]:
data.drop(columns=["Num Reviews"], axis=1, inplace=True)

Convert all `Course Code` to uppercase

In [55]:
data['Course Code'] = data['Course Code'].str.upper()

If the `Course Code` ends with a letter, remove it.
<br>
Example: CHEM111A --> Chem111

In [57]:
data['Course Code'] = data['Course Code'].str.replace(r'[A-Z]$', '', regex=True)

Remove rows where the Course Code is not given (empty `Course Code`)
<br>
There is only one empty course title, so this is feasible

In [59]:
data = data[data['Course Code'].str.strip() != '']

Let's use fuzzy matching NLP to group `Course Code` within each `department` that share at least a 92% similarity. <br>
This will standardize course codes to a single common course code, if they're similar enough

Example:<br>
CHEM111 and CHM111 --> CHEM111

In [61]:
from fuzzywuzzy import process

In [62]:
def similar_course_mapper(department):
    unique_courses = department['Course Code'].unique()
    course_map = {}
    
    for course in unique_courses:
        # find the closest matches to each course within the department
        matches = process.extract(course, unique_courses)
        # group courses together if they have a similarity score >= 92/100%
        similar_courses = [match for match, score in matches if score >= 92]
        
        if similar_courses:
            # if a similar course grouping exists --> the new course code will be the most popular course code of that group
            most_frequent_course = department[department['Course Code'].isin(similar_courses)]['Course Code'].mode()
            if len(most_frequent_course) > 0:
                most_frequent_course_value = most_frequent_course[0]
                for similar_course in similar_courses:
                    course_map[similar_course] = most_frequent_course_value
                    
    return course_map

In [63]:
# apply the function above to get the correct course code mappings within each department
course_corrections = data.groupby('Department').apply(similar_course_mapper, include_groups=False).to_dict()

In [64]:
# get the corrected course code for each row based on the department and class taken, otherwise default original course code
def correct_courses(row, corrections):
    return corrections.get(row['Department'], {}).get(row['Course Code'], row['Course Code'])

In [65]:
# apply the function above to correct the course codes
data['Course Code Corrected'] = data.apply(correct_courses, corrections=course_corrections, axis=1)

Only allow keep `course codes` that match the following format: [A-Z]+[0-9]{3,4}[A-Z]?

In other words, only keep `course codes` that start with alphabetical letters, followed by 3-4 digits, and an optional alphabetical letter indicating a type of course (i.e. T, M, A, S, D)

Examples:
Accepted Course Codes: CSE330S, CSE217A, CSE240
Not Accepted Course Codes: 100B, 100, B

In [67]:
course_code_pattern = r'^[A-Z]+[0-9]{3,4}[A-Z]?$'

print(data.shape)
data = data[data['Course Code'].str.contains(course_code_pattern, regex=True)]
print(data.shape)

(6443, 11)
(4857, 11)


In [68]:
data[["Course Code", "Course Code Corrected"]]

,Course Code,Course Code Corrected
0,MATH233,MATH233
1,MATH233,MATH233
2,MATH310,MATH310
3,MATH233,MATH233
4,MATH233,MATH233
...,...,...
6439,AMCS301,AMCS301
6442,GERMAN101,GERMAN101
6443,GERMAN101,GERMAN101
6444,MTH308,MTH308


Here's what the dataset looks like after cleaning

In [70]:
data.head()

,School_ID,School Name,Professor Name,Overall Quality,Department,Course Code,Review Date,Quality,Difficulty,Review Text,Course Code Corrected
0,1147,Washington University in St. Louis,Karl Schaefer,4.8,Mathematics,MATH233,2024-09-06,5.0,2.0,Best Professor I've Ever Had. He explains comp...,MATH233
1,1147,Washington University in St. Louis,Karl Schaefer,4.8,Mathematics,MATH233,2024-06-14,5.0,2.0,Probably one of the best math professors at wash,MATH233
2,1147,Washington University in St. Louis,Karl Schaefer,4.8,Mathematics,MATH310,2024-06-04,5.0,2.0,Schaefer is a very skilled educator and he is ...,MATH310
3,1147,Washington University in St. Louis,Karl Schaefer,4.8,Mathematics,MATH233,2024-06-04,5.0,2.0,Schaefer remains unparalleled for Calculus lec...,MATH233
4,1147,Washington University in St. Louis,Karl Schaefer,4.8,Mathematics,MATH233,2024-05-22,5.0,2.0,Professor Schaefer is an amazing Professor. He...,MATH233


Lets put this data tables; first lets make the normalized tables:

In [128]:
from sqlalchemy import create_engine, Column, Integer, String, Float, Date, ForeignKey, PrimaryKeyConstraint
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
import pandas as pd

# Use an in-memory SQLite database
engine = create_engine('sqlite:///:memory:')
Base = declarative_base()

# Define tables
class Schools(Base):
    __tablename__ = 'schools'
    SchoolID = Column(Integer, primary_key=True)
    SchoolName = Column(String, nullable=False)

class Departments(Base):
    __tablename__ = 'departments'
    DepartmentID = Column(Integer, primary_key=True)
    DepartmentName = Column(String, nullable=False)

class Professors(Base):
    __tablename__ = 'professors'
    ProfessorID = Column(Integer, primary_key=True)
    ProfessorName = Column(String, nullable=False)
    OverallQuality = Column(Float)

class Classes(Base):
    __tablename__ = 'classes'
    ClassID = Column(Integer, nullable=False)
    ProfessorID = Column(Integer, ForeignKey('professors.ProfessorID'), nullable=False)
    SchoolID = Column(Integer, ForeignKey('schools.SchoolID'), nullable=False)
    DepartmentID = Column(Integer, ForeignKey('departments.DepartmentID'), nullable=False)
    ClassName = Column(String, nullable=False)
    
    __table_args__ = (
        PrimaryKeyConstraint('ClassID', 'ProfessorID', name='class_professor_pk'),
    )

class ClassInstructors(Base):
    __tablename__ = 'class_instructors'
    ClassID = Column(Integer, ForeignKey('classes.ClassID'), primary_key=True, nullable=False)
    ProfessorID = Column(Integer, ForeignKey('professors.ProfessorID'), primary_key=True, nullable=False)

class Reviews(Base):
    __tablename__ = 'reviews'
    ReviewID = Column(Integer, primary_key=True, autoincrement=True)
    ClassID = Column(Integer, ForeignKey('classes.ClassID'), nullable=False)
    Quality = Column(Float, nullable=False)
    Difficulty = Column(Float, nullable=False)
    Review = Column(String)
    Date = Column(Date, nullable=False)

# Create all tables
Base.metadata.create_all(engine)

# Create a session to insert data
Session = sessionmaker(bind=engine)
session = Session()


C:\Users\Tengis\AppData\Local\Temp\ipykernel_34280\426450581.py:8: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


Then insert our cleaned data from the df

In [131]:
'''
# Initialize session
Session = sessionmaker(bind=engine)
session = Session()

# Insert unique Schools into Schools table with existence check
for school_id, school_name in data[['School_ID', 'School Name']].drop_duplicates().values:
    if not session.query(Schools).filter_by(SchoolID=school_id).first():
        session.add(Schools(SchoolID=school_id, SchoolName=school_name))

# Insert unique Departments into Departments table with existence check
for department_name in data['Department'].unique():
    if not session.query(Departments).filter_by(DepartmentName=department_name).first():
        session.add(Departments(DepartmentName=department_name))

# Insert unique Professors into Professors table with existence check
for professor_name, overall_quality in data[['Professor Name', 'Overall Quality']].drop_duplicates().values:
    if not session.query(Professors).filter_by(ProfessorName=professor_name).first():
        session.add(Professors(ProfessorName=professor_name, OverallQuality=overall_quality))

# Insert unique Classes into Classes table with existence check
for course_code, professor_name, school_id, department_name in data[['Course Code', 'Professor Name', 'School_ID', 'Department']].drop_duplicates().values:
    # Get the foreign key references for ProfessorID, SchoolID, and DepartmentID
    professor = session.query(Professors).filter_by(ProfessorName=professor_name).first()
    school = session.query(Schools).filter_by(SchoolID=school_id).first()
    department = session.query(Departments).filter_by(DepartmentName=department_name).first()

    if professor and school and department:
        if not session.query(Classes).filter_by(ClassName=course_code).first():
            session.add(Classes(ClassName=course_code, SchoolID=school.SchoolID, DepartmentID=department.DepartmentID))

# Commit changes before proceeding with foreign key relationships
session.commit()

# Insert into ClassInstructors table with existence check
for course_code, professor_name in data[['Course Code', 'Professor Name']].drop_duplicates().values:
    # Get the foreign key references for ClassID and ProfessorID
    class_record = session.query(Classes).filter_by(ClassName=course_code).first()
    professor = session.query(Professors).filter_by(ProfessorName=professor_name).first()

    if class_record and professor:
        if not session.query(ClassInstructors).filter_by(ClassID=class_record.ClassID, ProfessorID=professor.ProfessorID).first():
            session.add(ClassInstructors(ClassID=class_record.ClassID, ProfessorID=professor.ProfessorID))

# Commit final changes
session.commit()
'''

"\n# Initialize session\nSession = sessionmaker(bind=engine)\nsession = Session()\n\n# Insert unique Schools into Schools table with existence check\nfor school_id, school_name in data[['School_ID', 'School Name']].drop_duplicates().values:\n    if not session.query(Schools).filter_by(SchoolID=school_id).first():\n        session.add(Schools(SchoolID=school_id, SchoolName=school_name))\n\n# Insert unique Departments into Departments table with existence check\nfor department_name in data['Department'].unique():\n    if not session.query(Departments).filter_by(DepartmentName=department_name).first():\n        session.add(Departments(DepartmentName=department_name))\n\n# Insert unique Professors into Professors table with existence check\nfor professor_name, overall_quality in data[['Professor Name', 'Overall Quality']].drop_duplicates().values:\n    if not session.query(Professors).filter_by(ProfessorName=professor_name).first():\n        session.add(Professors(ProfessorName=professor_n

In [133]:
# Initialize session
Session = sessionmaker(bind=engine)
session = Session()

# Insert unique Schools into Schools table with existence check
for school_id, school_name in data[['School_ID', 'School Name']].drop_duplicates().values:
    if not session.query(Schools).filter_by(SchoolID=school_id).first():
        session.add(Schools(SchoolID=school_id, SchoolName=school_name))

# Insert unique Departments into Departments table with existence check
for department_name in data['Department'].unique():
    if not session.query(Departments).filter_by(DepartmentName=department_name).first():
        session.add(Departments(DepartmentName=department_name))

# Insert unique Professors into Professors table with existence check
for professor_name, overall_quality in data[['Professor Name', 'Overall Quality']].drop_duplicates().values:
    if not session.query(Professors).filter_by(ProfessorName=professor_name).first():
        session.add(Professors(ProfessorName=professor_name, OverallQuality=overall_quality))

# Commit changes before proceeding with foreign key relationships
session.commit()

# Insert unique Classes into Classes table with existence check
for course_code, professor_name, school_id, department_name in data[['Course Code', 'Professor Name', 'School_ID', 'Department']].drop_duplicates().values:
    # Get the foreign key references for ProfessorID, SchoolID, and DepartmentID
    professor = session.query(Professors).filter_by(ProfessorName=professor_name).first()
    school = session.query(Schools).filter_by(SchoolID=school_id).first()
    department = session.query(Departments).filter_by(DepartmentName=department_name).first()

    if professor and school and department:
        # Check if the class already exists
        existing_class = session.query(Classes).filter_by(ClassName=course_code, ProfessorID=professor.ProfessorID).first()
        if not existing_class:
            # Use a new ClassID that is the course code (or you can change this logic as needed)
            new_class = Classes(ClassID=course_code, ClassName=course_code, SchoolID=school.SchoolID, DepartmentID=department.DepartmentID, ProfessorID=professor.ProfessorID)
            session.add(new_class)

# Commit changes after inserting classes
session.commit()

# Insert into ClassInstructors table with existence check
for course_code, professor_name in data[['Course Code', 'Professor Name']].drop_duplicates().values:
    # Get the foreign key references for ClassID and ProfessorID
    class_record = session.query(Classes).filter_by(ClassName=course_code).first()
    professor = session.query(Professors).filter_by(ProfessorName=professor_name).first()

    if class_record and professor:
        if not session.query(ClassInstructors).filter_by(ClassID=class_record.ClassID, ProfessorID=professor.ProfessorID).first():
            session.add(ClassInstructors(ClassID=class_record.ClassID, ProfessorID=professor.ProfessorID))

# Commit changes after inserting class instructors
session.commit()

# Insert Reviews into the Reviews table
for course_code, review_date, quality, difficulty, review_text in data[['Course Code', 'Review Date', 'Quality', 'Difficulty', 'Review Text']].drop_duplicates().values:
    # Get the foreign key reference for ClassID
    class_record = session.query(Classes).filter_by(ClassName=course_code).first()

    if class_record:
        # Create a new Review record
        new_review = Reviews(ClassID=class_record.ClassID, Quality=quality, Difficulty=difficulty, Review=review_text, Date=pd.to_datetime(review_date).date())
        session.add(new_review)

# Commit final changes for reviews
session.commit()


Check to see if the data exists in the tables w/ some SQL

In [139]:
from sqlalchemy import text

# Function to query and print the existence of data in a table
def query_table_data(table_name, limit=10):
    with engine.connect() as connection:
        # Count the number of records in the table
        count_result = connection.execute(text(f"SELECT COUNT(*) FROM {table_name}"))
        count = count_result.fetchone()[0]
        print(f"Number of records in {table_name}: {count}")

        if count > 0:
            # If there are records, select a few to display
            print(f"Sample records from {table_name}:")
            result = connection.execute(text(f"SELECT * FROM {table_name} LIMIT {limit}"))
            for row in result:
                print(row)
        else:
            print(f"No records found in {table_name}.")
    
    print('------------------------------------')

# Query Schools table
query_table_data("schools")

# Query Departments table
query_table_data("departments")

# Query Professors table
query_table_data("professors")

# Query Classes table
query_table_data("classes")

# Query Reviews table
query_table_data("reviews")

# Query Class Instructors table
query_table_data("class_instructors")  # Added this line to include ClassInstructors


Number of records in schools: 1
Sample records from schools:
(1147, 'Washington University in St. Louis')
------------------------------------
Number of records in departments: 56
Sample records from departments:
(1, 'Mathematics')
(2, 'Chemistry')
(3, 'Psychology')
(4, 'Anthropology')
(5, 'Biology')
(6, 'English')
(7, 'Computer Science')
(8, 'Management')
(9, 'Marketing')
(10, 'History')
------------------------------------
Number of records in professors: 677
Sample records from professors:
(1, 'Karl Schaefer', 4.8)
(2, 'Bryn Lutes', 2.1)
(3, 'Silas Johnson', 4.5)
(4, 'Megan Daschbach', 4.9)
(5, 'Mitchell Sommers', 4.9)
(6, 'Glenn Stone', 3.8)
(7, 'Ian Duncan', 3.4)
(8, 'Jami Ake', 4.9)
(9, 'Richard Mabbs', 5.1)
(10, 'Karen Wooley', 3.0)
------------------------------------
Number of records in classes: 1968
Sample records from classes:
('MATH233', 1, 1147, 1, 'MATH233')
('MATH310', 1, 1147, 1, 'MATH310')
('MATH132', 1, 1147, 1, 'MATH132')
('MATH4351', 1, 1147, 1, 'MATH4351')
('MATH1